# Chapter 2: Build GPT From Scratch

Character-level GPT trained on tiny-Shakespeare. Every stage builds on the last — run cells in order.

See `../RUNBOOK.md` for the why/enterprise-notes/interview-angles that accompany this code.

## Stage 1: Data loading + character-level tokenization

In [ ]:
# Download the tiny Shakespeare dataset (~1MB)
!wget -nc https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print(f"Dataset length: {len(text):,} characters")
print("\nFirst 300 characters:\n")
print(text[:300])

### Build the vocabulary
Find every unique character in the dataset — that's our entire vocabulary. No subwords, no BPE merges yet (that's Chapter 4).

In [ ]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Vocabulary size: {vocab_size}")
print(f"All characters: {''.join(chars)!r}")

### Encoder / decoder
`stoi` (string-to-int) and `itos` (int-to-string) are the entire tokenizer. `encode` and `decode` are just dictionary lookups.

In [ ]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

test = "Hello"
encoded = encode(test)
print(f"{test!r} encoded: {encoded}")
print(f"Decoded back: {decode(encoded)!r}")

### Encode the entire dataset into a tensor

In [ ]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print(f"Data shape: {data.shape}, dtype: {data.dtype}")
print(f"First 100 encoded characters:\n{data[:100]}")

### Train / validation split (90 / 10)
Never evaluate on data the model trained on — validation loss is our only honest signal for overfitting.

In [ ]:
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Training characters:   {len(train_data):,}")
print(f"Validation characters: {len(val_data):,}")

### Context windows (`block_size`)
The model never sees the whole dataset at once — only a fixed-size window. One window of length `block_size` actually yields `block_size` separate training examples (predict the next char at every position).

In [ ]:
block_size = 8

x = train_data[:block_size]
y = train_data[1:block_size + 1]

print("One block of text yields these context -> target pairs:\n")
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"context: {decode(context.tolist())!r:20} -> target: {decode([target.item()])!r}")

### Batching
Stack many independent context windows together so the GPU processes them in parallel instead of one at a time.

In [ ]:
torch.manual_seed(1337)
batch_size = 4  # independent sequences processed in parallel

def get_batch(split):
    data_split = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_split) - block_size, (batch_size,))
    x = torch.stack([data_split[i:i+block_size] for i in ix])
    y = torch.stack([data_split[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:', xb.shape)
print(xb)
print('targets:', yb.shape)
print(yb)

**Checkpoint:** at this point you have `get_batch('train')` / `get_batch('val')` producing `(batch_size, block_size)` integer tensors ready to feed into embeddings. Next stage (Stage 2, in the next notebook section): token + position embeddings, and the first self-attention head.